# Lab 1 · EDA and simple baselines
Pairs · 24 September 2026, 12:30 Santiago · 6% NP.
Read the brief and rubric first. Supplied code supports execution; replace the written prompts and add your own analysis cells. MODE="synthetic" is fictional practice only. Real submission uses snapshot.

In [1]:
from pathlib import Path
import sys, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
for p in (Path.cwd(), *Path.cwd().parents):
    if (p/'labs/lab_01/lab01_support_v1.py').exists():
        ROOT=p; break
else:
    raise FileNotFoundError('Extract the full package and run inside it.')
sys.path.insert(0,str(ROOT/'labs/lab_01'))
from lab01_support_v1 import load_sources,merge_sources,majority_from_train,evaluate
MODE="snapshot"
print('DATA MODE:', MODE, '| Python:',platform.python_version(),'| pandas:',pd.__version__)

DATA MODE: snapshot | Python: 3.10.3 | pandas: 2.2.3


## 1. Target and audit
State the prediction moment, unit, population and target. Distinguish final classification from completing a race. Explain the source limitations. 

### Prediction moment

We will be predicting right before the race starts, that means that we'll check the 'grid' column to see where each driver started the race, so we'll be taking into account the penalties that could've been given between saturday and sunday.

### Unit

We'll be using a single race result from a single driver as the unit. In this case we'll be using a race from Leclerc.

### Population

The population are all the results achieved from Leclerc from 2019 to 2021, these are all the rows we have available for him provided by the snapshot. We'll be keeping all the results, whether he finished the race or not, was DQ or even missed qualifying.

### Target

The target is the final race position achieved by Charles Leclerc in a given race. Not 'finished' or 'points scored'. 

### Data limitation

The given data is a resume of the performance with what we know now, so that means that there could have been corrections made after some races that are present in the dataset but we wouldn't be really able to know what really happened. Maybe a 'points' = 0 row means that the driver finished in the top 10 but later received a penalty that made him end up in a lower position losing him the points. 

In [16]:
q_train,r_train=load_sources('train',ROOT,MODE)
train=merge_sources(q_train,r_train)
display(pd.DataFrame({'qualifying':q_train.groupby('season').size(),'results':r_train.groupby('season').size()}))
display(train.groupby('season').agg(rows=('driver_id','size'),missing_qualifying=('qualifying_position',lambda x:x.isna().sum())))
assert len(train)==len(r_train)
display(train['qualifying_match'].value_counts())

,qualifying,results
season,,
2019,418,420
2020,340,340
2021,439,440


,rows,missing_qualifying
season,,
2019,420,2
2020,340,0
2021,440,1


qualifying_match
both          1197
left_only        3
right_only       0
Name: count, dtype: int64

# Data integrity checks

We check only the results of Charles Leclerc, duplicates, missing values, qualifying-only records and coverage

In [68]:
# We keep only the results from Charles Leclerc
leclerc_races = train[(train['driver_id'] == 'leclerc')]
leclerc_races.reindex(columns=['season', 'race_name', 'driver_id', 'constructor_id', 'grid', 'position', 'position_text', 'status', 'qualifying_position', 'qualifying_match', 'target_top10'])


# We check for duplicates in the dataset based on the combination of 'season', 'race_name', and 'driver_id'
duplicates = leclerc_races.duplicated(subset=['season', 'race_name', 'driver_id'], keep=False)
if duplicates.any():
    print("Duplicate entries found for Charles Leclerc's races:")
    display(leclerc_races[duplicates])
    
else:
    print("No duplicate entries found for Charles Leclerc's races.")
    
# Qualifying-only records, where the driver was in qualy but not in the race
qualifying_only_records = leclerc_races[(leclerc_races['qualifying_position'].notnull() & leclerc_races['position_text'].isin(['R', 'W']))]
if qualifying_only_records.empty:
    print("No qualifying-only records found for Charles Leclerc's races.")
else:
    print("Qualifying-only records found for Charles Leclerc's races:")
    display(qualifying_only_records[['season', 'race_name', 'driver_id', 'qualifying_position', 'position_text', 'status']])

# Check for missing values in the dataset
missing_values = leclerc_races.isnull().sum()
if missing_values.any():
    print("Missing values found in Charles Leclerc's races:")
    display(missing_values[missing_values > 0])
else:
    print("No missing values found in Charles Leclerc's races.")
    
# Coverage by season for Charles Leclerc's races
coverage_by_season = leclerc_races.groupby('season').agg(
    total_races=('race_name', 'nunique'),
    races_with_results=('position_text', lambda x: x.ne('R').sum()),
    races_with_qualifying=('qualifying_position', lambda x: x.notnull().sum()),
)
print("Coverage by season for Charles Leclerc's races:")
display(coverage_by_season)


No duplicate entries found for Charles Leclerc's races.
Qualifying-only records found for Charles Leclerc's races:


,season,race_name,driver_id,qualifying_position,position_text,status
119,2019,Monaco Grand Prix,leclerc,16.0,R,Collision
216,2019,German Grand Prix,leclerc,10.0,R,Accident
458,2020,Styrian Grand Prix,leclerc,11.0,R,Collision damage
539,2020,Spanish Grand Prix,leclerc,9.0,R,Electronics
577,2020,Italian Grand Prix,leclerc,13.0,R,Accident
739,2020,Sakhir Grand Prix,leclerc,4.0,R,Collision
859,2021,Monaco Grand Prix,leclerc,1.0,W,Driveshaft
977,2021,Hungarian Grand Prix,leclerc,7.0,R,Collision


No missing values found in Charles Leclerc's races.
Coverage by season for Charles Leclerc's races:


,total_races,races_with_results,races_with_qualifying
season,,,
2019,21,19,21
2020,17,13,17
2021,22,21,22


# Why the left join instead of the inner join?

If we used an inner join instead of a left join starting from the results table, any race where Leclerc lacked a valid qualifying record would be entirely deleted rather than preserved with a null qualifying_position. This would artificially reduce the total_races count in our season coverage summary, erasing the record of any Grand Prix where he started or participated without setting a qualifying time. Crucially, dropping these rows violates the lab requirement to explicitly keep entries with missing qualifying records, thereby artificially filtering our retrospective population and breaking the baseline comparisons. 

## 2. EDA question 1
Question: TO COMPLETE.

Use train only. Add a table/graph, cite observed values, interpret and state a decision or limitation. At least one of the three analyses must check a Friday trap.

In [ ]:
# Add your train-only analysis here.

## 3. EDA question 2
Question: TO COMPLETE.

Use train only. Add a table/graph, cite observed values, interpret and state a decision or limitation. At least one of the three analyses must check a Friday trap.

In [ ]:
# Add your train-only analysis here.

## 4. EDA question 3
Question: TO COMPLETE.

Use train only. Add a table/graph, cite observed values, interpret and state a decision or limitation. At least one of the three analyses must check a Friday trap.

In [ ]:
# Add your train-only analysis here.

## 5. Baselines and 2022 check
Explain the two fixed rules, tie handling and missing-qualifying fallback. Predict qualitatively how each might fail before running the check. TO COMPLETE.

In [ ]:
majority=majority_from_train(train)
print('Training majority:',majority,'| training target rate:',train.target_top10.mean())
q_cal,r_cal=load_sources('calibration',ROOT,MODE)
cal=merge_sources(q_cal,r_cal)
print('Calibration rows:',len(cal),'| missing qualifying:',cal.qualifying_position.isna().sum())
display(evaluate(cal,majority))

Interpret TN/FP/FN/TP, accuracy and balanced accuracy. Explain the positive/negative error costs in this task without inventing monetary values. Record any corrections after the check. TO COMPLETE.

## 6. Freeze before final test
Complete this record and commit the code before enabling RUN_TEST.

- Members and date: TO COMPLETE.
- Target, population, join and prediction moment: TO COMPLETE.
- Baselines, train majority, threshold and missing policy: TO COMPLETE.
- Three analysis decisions and changes after calibration: TO COMPLETE.
- Frozen code commit/version: TO COMPLETE.

The test switch is a learning aid. Accessing CSVs directly still counts as inspecting test. Do not change the method in response to test results. Document genuine later bug fixes transparently.

In [ ]:
RUN_TEST=False
if RUN_TEST:
    q_test,r_test=load_sources('test',ROOT,MODE,frozen=True)
    test=merge_sources(q_test,r_test)
    print('Test rows:',len(test),'| missing qualifying:',test.qualifying_position.isna().sum())
    display(evaluate(test,majority))
else:
    print('Test remains unopened by this notebook. Complete and commit the freeze record first.')

## 7. Final interpretation and handoff
Compare calibration and test, cite your observed results, explain one failure mode and what the evidence supports. Discuss the retrospective population and source timing limitation. TO COMPLETE.

Complete RUNBOOK and PROMPTS/contributions; restart/run all and save outputs. Submit the repository URL and final commit via Canvas. No separate report is required.